**Reading the dataset & creating labels**

In [1]:
import os
from glob import glob
import pandas as pd

# CHANGE THIS
DATA_ROOT = "/kaggle/input/main-dataset-mri1/Dataset"  # contains train/ and test/

CLASS_MAP = {"healthy": 0, "tumor": 1}
IMG_EXTS = ("*.png", "*.jpg", "*.jpeg", "*.bmp")

def collect_split(split_name: str) -> pd.DataFrame:
    rows = []
    split_dir = os.path.join(DATA_ROOT, split_name)

    for cls_name, cls_id in CLASS_MAP.items():
        cls_dir = os.path.join(split_dir, cls_name)

        files = []
        for ext in IMG_EXTS:
            files.extend(glob(os.path.join(cls_dir, ext)))

        for fp in files:
            rows.append({"path": fp, "label": cls_id})

    df = pd.DataFrame(rows).sample(frac=1.0, random_state=42).reset_index(drop=True)
    return df

train_df = collect_split("train")
test_df  = collect_split("test")

print("Train samples:", len(train_df))
print("Test samples :", len(test_df))

print("\nTrain label counts:")
print(train_df["label"].value_counts())

print("\nTest label counts:")
print(test_df["label"].value_counts())

print("\nSample rows:")
print(train_df.head(5))


Train samples: 2214
Test samples : 394

Train label counts:
label
1    1147
0    1067
Name: count, dtype: int64

Test label counts:
label
1    254
0    140
Name: count, dtype: int64

Sample rows:
                                                path  label
0  /kaggle/input/main-dataset-mri1/Dataset/train/...      1
1  /kaggle/input/main-dataset-mri1/Dataset/train/...      1
2  /kaggle/input/main-dataset-mri1/Dataset/train/...      0
3  /kaggle/input/main-dataset-mri1/Dataset/train/...      1
4  /kaggle/input/main-dataset-mri1/Dataset/train/...      1


**Preprocess images (no augmentation yet)**

In [2]:
import tensorflow as tf
import numpy as np

IMG_SIZE = 224
BATCH_SIZE = 2
AUTOTUNE = tf.data.AUTOTUNE

def decode_image(path: tf.Tensor) -> tf.Tensor:
    """
    Reads an image from disk and decodes it into a float32 tensor in [0,1].
    Forces 3 channels for compatibility with ImageNet pretrained CNNs.
    """
    img_bytes = tf.io.read_file(path)
    img = tf.io.decode_image(img_bytes, channels=3, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)  # -> [0,1]
    return img

def letterbox_resize(img: tf.Tensor, target: int = 224) -> tf.Tensor:
    """
    Aspect-preserving resize + padding to avoid stretching/squeezing.
    Output shape: (target, target, 3)
    """
    shape = tf.shape(img)
    h = tf.cast(shape[0], tf.float32)
    w = tf.cast(shape[1], tf.float32)

    # Scale to fit within target x target
    scale = tf.minimum(target / h, target / w)
    new_h = tf.cast(tf.round(h * scale), tf.int32)
    new_w = tf.cast(tf.round(w * scale), tf.int32)

    img = tf.image.resize(img, (new_h, new_w), method="bilinear")

    # Compute padding amounts
    pad_h = target - new_h
    pad_w = target - new_w
    pad_top = pad_h // 2
    pad_bottom = pad_h - pad_top
    pad_left = pad_w // 2
    pad_right = pad_w - pad_left

    img = tf.pad(
        img,
        paddings=[[pad_top, pad_bottom], [pad_left, pad_right], [0, 0]],
        constant_values=0.0
    )

    # Ensure static shape known to the model
    img = tf.ensure_shape(img, [target, target, 3])
    return img

def preprocess_fn(path: tf.Tensor, label: tf.Tensor):
    """
    Full preprocessing: decode -> letterbox resize.
    (No augmentation here; )
    """
    img = decode_image(path)
    img = letterbox_resize(img, IMG_SIZE)
    return img, label

def make_dataset(df, batch_size=2, shuffle=False):
    paths = df["path"].astype(str).values
    labels = df["label"].astype(np.int32).values

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(df), 2000), seed=42, reshuffle_each_iteration=True)

    ds = ds.map(preprocess_fn, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds

# Build preprocessed datasets
train_ds_pre = make_dataset(train_df, batch_size=BATCH_SIZE, shuffle=False)
test_ds_pre  = make_dataset(test_df, batch_size=BATCH_SIZE, shuffle=False)

# Quick sanity check: one batch
for images, labels in train_ds_pre.take(1):
    print("Batch images shape:", images.shape)   # (B, 224, 224, 3)
    print("Batch labels shape:", labels.shape)   # (B,)
    print("Pixel range:", float(tf.reduce_min(images)), "to", float(tf.reduce_max(images)))


2026-02-07 00:25:09.943211: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770423910.116936      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770423910.165014      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770423910.577331      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770423910.577370      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770423910.577372      55 computation_placer.cc:177] computation placer alr

Batch images shape: (2, 224, 224, 3)
Batch labels shape: (2,)
Pixel range: 0.0 to 0.9752107858657837


**light augmentation module**

In [3]:
import tensorflow as tf

# Light augmentation (MRI-safe)
augment = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(factor=0.03),      # ~±10 degrees
    tf.keras.layers.RandomTranslation(0.05, 0.05),    # up to 5%
    tf.keras.layers.RandomZoom(0.05),                 # ±5%
    tf.keras.layers.RandomContrast(0.05),             # mild
], name="light_augment")


**augmentation to the TRAIN dataset only**

In [4]:
AUTOTUNE = tf.data.AUTOTUNE

def apply_augment(img, label):
    # training=True makes randomness active
    img = augment(img, training=True)
    return img, label

train_ds_aug = train_ds_pre.map(apply_augment, num_parallel_calls=AUTOTUNE)
test_ds_aug  = test_ds_pre  # test stays unchanged


**Sanity check (shapes + ranges)**

In [5]:
for images, labels in train_ds_aug.take(1):
    print("Aug Train batch:", images.shape, labels.shape)
    print("Pixel range:", float(tf.reduce_min(images)), "to", float(tf.reduce_max(images)))

for images, labels in test_ds_aug.take(1):
    print("Test batch (no aug):", images.shape, labels.shape)


Aug Train batch: (2, 224, 224, 3) (2,)
Pixel range: 0.0 to 0.8658542633056641
Test batch (no aug): (2, 224, 224, 3) (2,)


**3 CNN feature extractors (as embedding models, not classifiers)**

In [6]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import ResNet50, VGG16, EfficientNetB0
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_pre
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_pre
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre

# 1) Build extractor models (include_top=False + GAP)
def build_resnet_extractor(img_size=224):
    base = ResNet50(weights="imagenet", include_top=False, input_shape=(img_size, img_size, 3))
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    return Model(base.input, x, name="ResNet50_Extractor")

def build_vgg_extractor(img_size=224):
    base = VGG16(weights="imagenet", include_top=False, input_shape=(img_size, img_size, 3))
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    return Model(base.input, x, name="VGG16_Extractor")

def build_effnet_extractor(img_size=224):
    base = EfficientNetB0(weights="imagenet", include_top=False, input_shape=(img_size, img_size, 3))
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    return Model(base.input, x, name="EfficientNetB0_Extractor")

resnet_extractor = build_resnet_extractor()
vgg_extractor    = build_vgg_extractor()
eff_extractor    = build_effnet_extractor()

print(resnet_extractor.name, "output:", resnet_extractor.output_shape)
print(vgg_extractor.name,    "output:", vgg_extractor.output_shape)
print(eff_extractor.name,    "output:", eff_extractor.output_shape)

# 2) Sanity check on one batch (use your augmented train dataset)
for images, labels in train_ds_aug.take(1):
    # Important: each backbone expects its own preprocess_input
    r = resnet_extractor(resnet_pre(images), training=False)
    v = vgg_extractor(vgg_pre(images), training=False)
    e = eff_extractor(eff_pre(images), training=False)

    print("\nInput batch:", images.shape)
    print("ResNet feats:", r.shape)  # (B, 2048)
    print("VGG feats   :", v.shape)  # (B, 512)
    print("EffNet feats:", e.shape)  # (B, 1280)


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
ResNet50_Extractor output: (None, 2048)
VGG16_Extractor output: (None, 512)
EfficientNetB0_Extractor output: (None, 1280)


I0000 00:00:1770423929.478518      55 cuda_dnn.cc:529] Loaded cuDNN version 91002



Input batch: (2, 224, 224, 3)
ResNet feats: (2, 2048)
VGG feats   : (2, 512)
EffNet feats: (2, 1280)


**Full feature extraction + feature fusion**

In [7]:
import numpy as np
import tensorflow as tf

def extract_fused_features(ds, resnet_extractor, vgg_extractor, eff_extractor,
                          resnet_pre, vgg_pre, eff_pre, verbose=True):
    """
    Takes a tf.data.Dataset yielding (images, labels) and returns:
      X_fused: (N, 3840) numpy array
      y:      (N,) numpy array
    """
    fused_chunks = []
    y_chunks = []

    total_batches = 0
    for _ in ds:
        total_batches += 1

    # Iterate again (datasets are re-iterable)
    for bi, (images, labels) in enumerate(ds, start=1):
        # Apply backbone-specific preprocessing + extract embeddings
        r = resnet_extractor(resnet_pre(images), training=False)  # (B, 2048)
        v = vgg_extractor(vgg_pre(images), training=False)        # (B, 512)
        e = eff_extractor(eff_pre(images), training=False)        # (B, 1280)

        # Concatenate along feature dimension
        fused = tf.concat([r, v, e], axis=1)                      # (B, 3840)

        fused_chunks.append(fused.numpy())
        y_chunks.append(labels.numpy())

        if verbose and (bi % 20 == 0 or bi == total_batches):
            print(f"Processed batch {bi}/{total_batches}")

    X_fused = np.vstack(fused_chunks).astype(np.float32)
    y = np.concatenate(y_chunks).astype(np.int32)
    return X_fused, y

# --- Train (augmented) features ---
X_train_fused, y_train_out = extract_fused_features(
    train_ds_aug,
    resnet_extractor, vgg_extractor, eff_extractor,
    resnet_pre, vgg_pre, eff_pre,
    verbose=True
)

# --- Test (no augmentation) features ---
X_test_fused, y_test_out = extract_fused_features(
    test_ds_aug,
    resnet_extractor, vgg_extractor, eff_extractor,
    resnet_pre, vgg_pre, eff_pre,
    verbose=True
)

print("\nFinal shapes:")
print("X_train_fused:", X_train_fused.shape, "y_train:", y_train_out.shape)
print("X_test_fused :", X_test_fused.shape,  "y_test :", y_test_out.shape)

# Sanity check: feature dimension must be 3840
assert X_train_fused.shape[1] == 3840, "Train fused feature dim is not 3840!"
assert X_test_fused.shape[1] == 3840,  "Test fused feature dim is not 3840!"

# Optional: save to disk so you never recompute features
np.save("X_train_fused.npy", X_train_fused)
np.save("y_train.npy", y_train_out)
np.save("X_test_fused.npy", X_test_fused)
np.save("y_test.npy", y_test_out)

print("\nSaved: X_train_fused.npy, y_train.npy, X_test_fused.npy, y_test.npy")


Processed batch 20/1107
Processed batch 40/1107
Processed batch 60/1107
Processed batch 80/1107
Processed batch 100/1107
Processed batch 120/1107
Processed batch 140/1107
Processed batch 160/1107
Processed batch 180/1107
Processed batch 200/1107
Processed batch 220/1107
Processed batch 240/1107
Processed batch 260/1107
Processed batch 280/1107
Processed batch 300/1107
Processed batch 320/1107
Processed batch 340/1107
Processed batch 360/1107
Processed batch 380/1107
Processed batch 400/1107
Processed batch 420/1107
Processed batch 440/1107
Processed batch 460/1107
Processed batch 480/1107
Processed batch 500/1107
Processed batch 520/1107
Processed batch 540/1107
Processed batch 560/1107
Processed batch 580/1107
Processed batch 600/1107
Processed batch 620/1107
Processed batch 640/1107
Processed batch 660/1107
Processed batch 680/1107
Processed batch 700/1107
Processed batch 720/1107
Processed batch 740/1107
Processed batch 760/1107
Processed batch 780/1107
Processed batch 800/1107
Proc

**Load saved features**

In [8]:
import numpy as np
from sklearn.preprocessing import StandardScaler

X_train_fused = np.load("X_train_fused.npy")
y_train = np.load("y_train.npy")

X_test_fused = np.load("X_test_fused.npy")
y_test = np.load("y_test.npy")

print("Loaded shapes:")
print("X_train:", X_train_fused.shape)
print("X_test :", X_test_fused.shape)


Loaded shapes:
X_train: (2214, 3840)
X_test : (394, 3840)


**Fit scaler on training features**

In [9]:
scaler = StandardScaler()

# Fit ONLY on training data
X_train_scaled = scaler.fit_transform(X_train_fused)

# Apply same transformation to test data
X_test_scaled = scaler.transform(X_test_fused)


**Sanity check scaling**

In [10]:
print("Train mean (first 5 features):", X_train_scaled.mean(axis=0)[:5])
print("Train std  (first 5 features):", X_train_scaled.std(axis=0)[:5])

print("Test mean  (first 5 features):", X_test_scaled.mean(axis=0)[:5])
print("Test std   (first 5 features):", X_test_scaled.std(axis=0)[:5])


Train mean (first 5 features): [ 0.0000000e+00  2.1986448e-07  2.6278272e-07 -1.4621176e-07
 -2.5844833e-09]
Train std  (first 5 features): [0.         1.0000149  0.99999875 1.0000039  0.9999969 ]
Test mean  (first 5 features): [ 0.         -0.02125739 -0.03763575 -0.07304181  0.05221527]
Test std   (first 5 features): [0.0000000e+00 3.3527613e-08 5.7772762e-01 1.7881393e-07 1.2120376e+00]


In [11]:
np.save("X_train_scaled.npy", X_train_scaled)
np.save("X_test_scaled.npy", X_test_scaled)

print("Saved: X_train_scaled.npy, X_test_scaled.npy")


Saved: X_train_scaled.npy, X_test_scaled.npy


In [12]:
import numpy as np

# -------------------------
# Load scaled features
# -------------------------
X_train = np.load("X_train_scaled.npy").astype(np.float32)  # (2214, 3840)
X_test  = np.load("X_test_scaled.npy").astype(np.float32)   # (394, 3840)
y_train = np.load("y_train.npy").astype(np.float32)         # (2214,)

N_samples, D = X_train.shape
print("Loaded:", X_train.shape, X_test.shape, y_train.shape)

# -------------------------
# Utility: correlation
# Corr(X,Y) = cov(X,Y) / (stdX*stdY)
# We compute correlations using training data only.
# -------------------------
eps = 1e-12

# Standardize y for correlation computation (Q in the paper)
y = y_train.copy()
y = y - y.mean()
std_y = y.std() + eps
y = y / std_y  # now mean 0, std 1

# X_train is already standardized by StandardScaler (mean~0, std~1 per feature)
# but some features can be constant -> std=0 -> column is all zeros.
# That's OK; correlations will become ~0 safely.

def binarize_position(pos, threshold=0.5):
    """Convert continuous position vector to a binary mask."""
    return (pos >= threshold).astype(np.uint8)

def ensure_nonempty(mask, rng):
    """Ensure at least one feature is selected."""
    if mask.sum() == 0:
        mask[rng.integers(0, mask.size)] = 1
    return mask

def fitness_eq2(mask):
    """
    EXACT fitness from Eq. (2):

    Fitness(x) = [ (1/K^2) * sum_{i=1..K} sum_{j=1..K} |Corr(x_i, x_j)| ]
                 / [ sqrt(K) * ( 1 + (1/K) * sum_{k=1..K} |Corr(x_k, Q)| ) ]

    where x_i are the selected feature columns, Q is the label vector.
    """
    idx = np.flatnonzero(mask)
    K = idx.size
    if K == 0:
        return np.inf

    Xs = X_train[:, idx]  # (N_samples, K)

    # --- Corr(feature_k, Q) for each selected feature (absolute) ---
    # Since y is mean0 std1, corr(x, y) = cov(x,y)/(std_x*1)
    # But std_x is ~1 for standardized features; for constant columns it is 0 -> handle with eps.
    # Compute cov(x,y) = mean(x*y) because both are mean0 (approximately).
    # More exactly: cov = (x_centered·y_centered)/(N-1). Here y is centered; X is centered by scaler.
    # We'll compute using (N-1) for correctness.
    denom = float(N_samples - 1)
    # std_x per selected feature:
    std_x = Xs.std(axis=0) + eps
    cov_xy = (Xs * y[:, None]).sum(axis=0) / denom
    corr_xq = np.abs(cov_xy / std_x)  # since std_y = 1 after normalization

    # --- Corr among selected features (absolute) ---
    # Compute correlation matrix Corr(Xs_i, Xs_j) exactly:
    # Corr = Cov / (std_i * std_j)
    # Cov matrix = (Xs^T Xs)/(N-1) because Xs approx mean0
    cov_mat = (Xs.T @ Xs) / denom  # (K,K)
    denom_mat = (std_x[:, None] * std_x[None, :]) + eps
    corr_mat = cov_mat / denom_mat

    numerator = (np.abs(corr_mat).sum()) / (K * K)  # includes diagonal (|1| terms) as in Eq(2)
    denominator = (np.sqrt(K) * (1.0 + (corr_xq.mean())))  # 1 + (1/K) sum |Corr(xk,Q)|

    return float(numerator / (denominator + eps))

# -------------------------
# BHO (Steps 1–9) EXACT
# -------------------------
def bho_select_features(
    pop_size=25,          # N in the paper
    iterations=80,        # T in the paper
    threshold=0.5,        # for mapping position -> binary
    seed=42
):
    rng = np.random.default_rng(seed)

    # Step 1: Create starting population of binary strings (solution vectors) at random.
    # We'll store positions as float in {0,1} initially (still binary).
    pop_pos = rng.integers(0, 2, size=(pop_size, D)).astype(np.float32)  # 0/1
    pop_mask = np.zeros((pop_size, D), dtype=np.uint8)
    pop_fit = np.zeros(pop_size, dtype=np.float64)

    for i in range(pop_size):
        pop_mask[i] = ensure_nonempty(pop_pos[i].astype(np.uint8), rng)
        pop_pos[i] = pop_mask[i].astype(np.float32)
        # Step 2: Determine each solution vector’s fitness using Eq.(2).
        pop_fit[i] = fitness_eq2(pop_mask[i])

    # Step 3: Determine black hole with the lowest fitness.
    bh_idx = int(np.argmin(pop_fit))
    XBH = pop_pos[bh_idx].copy()
    BH_mask = pop_mask[bh_idx].copy()
    BH_fit = float(pop_fit[bh_idx])

    history = [BH_fit]

    for t in range(iterations):
        # Step 4: Adjust each solution’s location using Eq.(3)
        for i in range(pop_size):
            if i == bh_idx:
                continue

            # Eq.(3): Xi = Xi + rand * (XBH - Xi)
            rand_scalar = float(rng.random())  # rand in (0,1) (scalar, as written)
            pop_pos[i] = pop_pos[i] + rand_scalar * (XBH - pop_pos[i])

            # Convert updated position to binary string for fitness evaluation
            pop_mask[i] = binarize_position(pop_pos[i], threshold=threshold).astype(np.uint8)
            pop_mask[i] = ensure_nonempty(pop_mask[i], rng)
            pop_fit[i] = fitness_eq2(pop_mask[i])

        # Step 5: Calculate threshold distance R (event horizon) using Eq.(4)
        denom_sum = float(np.sum(pop_fit) + eps)
        R = BH_fit / denom_sum

        # Step 6: If any solution has lower fitness than BH, replace BH with it.
        new_bh_idx = int(np.argmin(pop_fit))
        if float(pop_fit[new_bh_idx]) < BH_fit:
            bh_idx = new_bh_idx
            XBH = pop_pos[bh_idx].copy()
            BH_mask = pop_mask[bh_idx].copy()
            BH_fit = float(pop_fit[bh_idx])

        # Step 7: Remove solutions within distance R from BH and replace with new random solution vectors.
        # Paper says "within threshold distance R from black hole" — distance in solution space.
        # We use Euclidean distance in the (continuous) position space.
        for i in range(pop_size):
            if i == bh_idx:
                continue
            dist = float(np.linalg.norm(pop_pos[i] - XBH))  # Euclidean distance
            if dist < R:
                # replace with randomly generated binary solution vector
                pop_pos[i] = rng.integers(0, 2, size=D).astype(np.float32)
                pop_mask[i] = ensure_nonempty(pop_pos[i].astype(np.uint8), rng)
                pop_pos[i] = pop_mask[i].astype(np.float32)
                pop_fit[i] = fitness_eq2(pop_mask[i])

        # Step 8: repeat until iterations reached
        history.append(BH_fit)

        if (t + 1) % 10 == 0 or (t + 1) == iterations:
            print(f"Iter {t+1:03d}/{iterations} | BH fitness={BH_fit:.6f} | selected K={int(BH_mask.sum())}")

    # Step 9: Return BH with lowest fitness
    selected_idx = np.flatnonzero(BH_mask)
    return BH_mask, selected_idx, np.array(history, dtype=np.float64)

# -------------------------
# Run BHO
# -------------------------
best_mask, selected_idx, bho_history = bho_select_features(
    pop_size=25,
    iterations=80,
    threshold=0.5,
    seed=42
)

print("\nBHO done.")
print("Selected K =", selected_idx.size, "out of", D)

# Apply to train/test (deliverables)
X_train_bho = X_train[:, selected_idx]
X_test_bho  = X_test[:, selected_idx]

print("X_train_bho:", X_train_bho.shape)
print("X_test_bho :", X_test_bho.shape)

# Save outputs
np.save("bho_best_mask.npy", best_mask)
np.save("bho_selected_idx.npy", selected_idx)
np.save("X_train_bho.npy", X_train_bho)
np.save("X_test_bho.npy", X_test_bho)
np.save("bho_history.npy", bho_history)

print("\nSaved bho_best_mask.npy, bho_selected_idx.npy, X_train_bho.npy, X_test_bho.npy, bho_history.npy")


Loaded: (2214, 3840) (394, 3840) (2214,)
Iter 010/80 | BH fitness=0.002212 | selected K=1958
Iter 020/80 | BH fitness=0.002121 | selected K=1938
Iter 030/80 | BH fitness=0.002117 | selected K=1965
Iter 040/80 | BH fitness=0.002117 | selected K=1965
Iter 050/80 | BH fitness=0.002117 | selected K=1965
Iter 060/80 | BH fitness=0.002084 | selected K=1997
Iter 070/80 | BH fitness=0.002084 | selected K=1997
Iter 080/80 | BH fitness=0.002067 | selected K=2030

BHO done.
Selected K = 2030 out of 3840
X_train_bho: (2214, 2030)
X_test_bho : (394, 2030)

Saved bho_best_mask.npy, bho_selected_idx.npy, X_train_bho.npy, X_test_bho.npy, bho_history.npy



**Baselines to run: Logistic Regression (L2), Linear SVM, RBF SVM, Random Forest**


In [13]:
import numpy as np

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import RandomForestClassifier

# -----------------------
# 1) Load data
# -----------------------
X_train = np.load("X_train_bho.npy").astype(np.float32)   # (2214, K)
X_test  = np.load("X_test_bho.npy").astype(np.float32)    # (394, K)
y_train = np.load("y_train.npy").astype(np.int32)         # (2214,)
y_test  = np.load("y_test.npy").astype(np.int32)          # (394,)

print("Shapes:", X_train.shape, X_test.shape, y_train.shape, y_test.shape)

# -----------------------
# 2) Helpers
# -----------------------
def evaluate_model(name, model, Xtr, ytr, Xte, yte):
    model.fit(Xtr, ytr)
    y_pred = model.predict(Xte)

    acc = accuracy_score(yte, y_pred)
    prec = precision_score(yte, y_pred, zero_division=0)
    rec = recall_score(yte, y_pred, zero_division=0)
    f1 = f1_score(yte, y_pred, zero_division=0)
    cm = confusion_matrix(yte, y_pred)

    # AUROC: use predict_proba if available, else decision_function if available
    auc = None
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(Xte)[:, 1]
        auc = roc_auc_score(yte, y_score)
    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(Xte)
        auc = roc_auc_score(yte, y_score)

    print(f"\n=== {name} ===")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1       : {f1:.4f}")
    if auc is not None:
        print(f"AUROC    : {auc:.4f}")
    else:
        print("AUROC    : (not available)")
    print("Confusion Matrix:\n", cm)

    return {
        "model": name,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "auroc": auc
    }

# -----------------------
# 3) Baseline models
# -----------------------
models = [
    ("LogReg (L2)", LogisticRegression(max_iter=5000, n_jobs=None)),
    ("Linear SVM",  LinearSVC()),  # fast; uses decision_function for AUROC
    ("RBF SVM",     SVC(kernel="rbf", probability=True)),  # slower; gives predict_proba
    ("RandomForest", RandomForestClassifier(
        n_estimators=400, random_state=42, n_jobs=-1
    )),
]

results = []
for name, model in models:
    results.append(evaluate_model(name, model, X_train, y_train, X_test, y_test))

# -----------------------
# 4) Print compact summary
# -----------------------
print("\n\n===== Summary (test) =====")
for r in results:
    print(
        f"{r['model']:<12} | acc={r['accuracy']:.4f} "
        f"f1={r['f1']:.4f} auc={None if r['auroc'] is None else round(r['auroc'], 4)}"
    )


Shapes: (2214, 2030) (394, 2030) (2214,) (394,)

=== LogReg (L2) ===
Accuracy : 0.9670
Precision: 0.9918
Recall   : 0.9567
F1       : 0.9739
AUROC    : 0.9905
Confusion Matrix:
 [[138   2]
 [ 11 243]]

=== Linear SVM ===
Accuracy : 0.9721
Precision: 0.9728
Recall   : 0.9843
F1       : 0.9785
AUROC    : 0.9694
Confusion Matrix:
 [[133   7]
 [  4 250]]

=== RBF SVM ===
Accuracy : 0.9416
Precision: 1.0000
Recall   : 0.9094
F1       : 0.9526
AUROC    : 0.9990
Confusion Matrix:
 [[140   0]
 [ 23 231]]

=== RandomForest ===
Accuracy : 0.9391
Precision: 1.0000
Recall   : 0.9055
F1       : 0.9504
AUROC    : 0.9983
Confusion Matrix:
 [[140   0]
 [ 24 230]]


===== Summary (test) =====
LogReg (L2)  | acc=0.9670 f1=0.9739 auc=0.9905
Linear SVM   | acc=0.9721 f1=0.9785 auc=0.9694
RBF SVM      | acc=0.9416 f1=0.9526 auc=0.999
RandomForest | acc=0.9391 f1=0.9504 auc=0.9983


**Comparision with and without BHO Feature selection**

In [14]:
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

# -----------------------
# Load WITH BHO
# -----------------------
Xtr_bho = np.load("X_train_bho.npy").astype(np.float32)
Xte_bho = np.load("X_test_bho.npy").astype(np.float32)

# -----------------------
# Load WITHOUT BHO
# -----------------------
Xtr_full = np.load("X_train_scaled.npy").astype(np.float32)
Xte_full = np.load("X_test_scaled.npy").astype(np.float32)

# Labels
y_train = np.load("y_train.npy").astype(np.int32)
y_test  = np.load("y_test.npy").astype(np.int32)

print("With BHO :", Xtr_bho.shape, Xte_bho.shape)
print("No BHO   :", Xtr_full.shape, Xte_full.shape)


With BHO : (2214, 2030) (394, 2030)
No BHO   : (2214, 3840) (394, 3840)


In [15]:
def evaluate(model, Xtr, ytr, Xte, yte):
    model.fit(Xtr, ytr)
    y_pred = model.predict(Xte)

    acc = accuracy_score(yte, y_pred)
    prec = precision_score(yte, y_pred, zero_division=0)
    rec = recall_score(yte, y_pred, zero_division=0)
    f1 = f1_score(yte, y_pred, zero_division=0)

    auc = None
    if hasattr(model, "predict_proba"):
        auc = roc_auc_score(yte, model.predict_proba(Xte)[:, 1])
    elif hasattr(model, "decision_function"):
        auc = roc_auc_score(yte, model.decision_function(Xte))

    return acc, f1, auc


**Run comparison (Linear SVM + Logistic Regression)**

In [16]:
models = {
    "LogReg": LogisticRegression(max_iter=5000),
    "LinearSVM": LinearSVC()
}

print("\n===== WITH vs WITHOUT BHO =====")
for name, model in models.items():

    acc_bho, f1_bho, auc_bho = evaluate(
        model, Xtr_bho, y_train, Xte_bho, y_test
    )

    acc_full, f1_full, auc_full = evaluate(
        model, Xtr_full, y_train, Xte_full, y_test
    )

    print(f"\n{name}")
    print(f" WITH BHO    | acc={acc_bho:.4f} f1={f1_bho:.4f} auc={auc_bho:.4f}")
    print(f" WITHOUT BHO | acc={acc_full:.4f} f1={f1_full:.4f} auc={auc_full:.4f}")



===== WITH vs WITHOUT BHO =====

LogReg
 WITH BHO    | acc=0.9670 f1=0.9739 auc=0.9905
 WITHOUT BHO | acc=0.9695 f1=0.9761 auc=0.9976

LinearSVM
 WITH BHO    | acc=0.9721 f1=0.9785 auc=0.9694
 WITHOUT BHO | acc=0.9645 f1=0.9723 auc=0.9778


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [18]:
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

models = {
    "LogReg": LogisticRegression(max_iter=5000),
    "LinearSVM": LinearSVC(),
    "RBF_SVM": SVC(kernel="rbf", probability=True),
    "RandomForest": RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1),
    "GaussianNB": GaussianNB()
}

print("\n===== WITH vs WITHOUT BHO (5 classifiers) =====")
for name, model in models.items():
    acc_bho, f1_bho, auc_bho = evaluate(model, Xtr_bho, y_train, Xte_bho, y_test)
    acc_full, f1_full, auc_full = evaluate(model, Xtr_full, y_train, Xte_full, y_test)

    print(f"\n{name}")
    print(f" WITH BHO    | acc={acc_bho:.4f} f1={f1_bho:.4f} auc={auc_bho if auc_bho is None else round(auc_bho, 4)}")
    print(f" WITHOUT BHO | acc={acc_full:.4f} f1={f1_full:.4f} auc={auc_full if auc_full is None else round(auc_full, 4)}")


===== WITH vs WITHOUT BHO (5 classifiers) =====

LogReg
 WITH BHO    | acc=0.9670 f1=0.9739 auc=0.9905
 WITHOUT BHO | acc=0.9695 f1=0.9761 auc=0.9976


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(



LinearSVM
 WITH BHO    | acc=0.9721 f1=0.9785 auc=0.9694
 WITHOUT BHO | acc=0.9645 f1=0.9723 auc=0.9785

RBF_SVM
 WITH BHO    | acc=0.9416 f1=0.9526 auc=0.999
 WITHOUT BHO | acc=0.9518 f1=0.9611 auc=0.9993

RandomForest
 WITH BHO    | acc=0.9391 f1=0.9504 auc=0.9983
 WITHOUT BHO | acc=0.9442 f1=0.9547 auc=0.9986

GaussianNB
 WITH BHO    | acc=0.6853 f1=0.7786 auc=0.6181
 WITHOUT BHO | acc=0.6751 f1=0.7638 auc=0.6175


In [19]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import (
    RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier,
    ExtraTreesClassifier
)
from sklearn.naive_bayes import GaussianNB

# -----------------------
# Optional: XGBoost
# -----------------------
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception as e:
    HAS_XGB = False
    print("XGBoost not available (skipping XGBClassifier). Error:", e)

# -----------------------
# Load WITH BHO
# -----------------------
Xtr_bho = np.load("X_train_bho.npy").astype(np.float32)
Xte_bho = np.load("X_test_bho.npy").astype(np.float32)

# -----------------------
# Load WITHOUT BHO
# -----------------------
Xtr_full = np.load("X_train_scaled.npy").astype(np.float32)
Xte_full = np.load("X_test_scaled.npy").astype(np.float32)

# Labels
y_train = np.load("y_train.npy").astype(np.int32)
y_test  = np.load("y_test.npy").astype(np.int32)

print("With BHO :", Xtr_bho.shape, Xte_bho.shape)
print("No BHO   :", Xtr_full.shape, Xte_full.shape)

# -----------------------
# Helper
# -----------------------
def evaluate(model, Xtr, ytr, Xte, yte):
    model.fit(Xtr, ytr)
    y_pred = model.predict(Xte)

    acc = accuracy_score(yte, y_pred)
    f1 = f1_score(yte, y_pred, zero_division=0)

    auc = None
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(Xte)[:, 1]
        auc = roc_auc_score(yte, y_score)
    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(Xte)
        auc = roc_auc_score(yte, y_score)

    return acc, f1, auc

# -----------------------
# Baseline models
# -----------------------
models = {
    # Linear baselines
    "LogReg": LogisticRegression(max_iter=5000),
    "LinearSVM": LinearSVC(),

    # Kernel baseline
    "RBF_SVM": SVC(kernel="rbf", probability=True),

    # Probabilistic sanity baseline
    "GaussianNB": GaussianNB(),

    # Tree ensembles
    "RandomForest": RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1),
    "ExtraTrees": ExtraTreesClassifier(n_estimators=600, random_state=42, n_jobs=-1),

    # Boosting
    "AdaBoost": AdaBoostClassifier(
        n_estimators=400,
        learning_rate=0.5,
        random_state=42
    ),
    "GradBoost": GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ),
}

# XGBoost (if available)
if HAS_XGB:
    models["XGBoost"] = XGBClassifier(
        n_estimators=600,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1
    )

# -----------------------
# Run WITH vs WITHOUT BHO
# -----------------------
print("\n===== WITH vs WITHOUT BHO (extended baselines) =====")
for name, model in models.items():
    acc_bho, f1_bho, auc_bho = evaluate(model, Xtr_bho, y_train, Xte_bho, y_test)
    acc_full, f1_full, auc_full = evaluate(model, Xtr_full, y_train, Xte_full, y_test)

    print(f"\n{name}")
    print(f" WITH BHO    | acc={acc_bho:.4f} f1={f1_bho:.4f} auc={auc_bho if auc_bho is None else round(auc_bho, 4)}")
    print(f" WITHOUT BHO | acc={acc_full:.4f} f1={f1_full:.4f} auc={auc_full if auc_full is None else round(auc_full, 4)}")


With BHO : (2214, 2030) (394, 2030)
No BHO   : (2214, 3840) (394, 3840)

===== WITH vs WITHOUT BHO (extended baselines) =====

LogReg
 WITH BHO    | acc=0.9670 f1=0.9739 auc=0.9905
 WITHOUT BHO | acc=0.9695 f1=0.9761 auc=0.9976


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(



LinearSVM
 WITH BHO    | acc=0.9721 f1=0.9785 auc=0.9694
 WITHOUT BHO | acc=0.9645 f1=0.9723 auc=0.9778

RBF_SVM
 WITH BHO    | acc=0.9416 f1=0.9526 auc=0.999
 WITHOUT BHO | acc=0.9518 f1=0.9611 auc=0.9993

GaussianNB
 WITH BHO    | acc=0.6853 f1=0.7786 auc=0.6181
 WITHOUT BHO | acc=0.6751 f1=0.7638 auc=0.6175

RandomForest
 WITH BHO    | acc=0.9391 f1=0.9504 auc=0.9983
 WITHOUT BHO | acc=0.9442 f1=0.9547 auc=0.9986

ExtraTrees
 WITH BHO    | acc=0.9594 f1=0.9675 auc=0.9987
 WITHOUT BHO | acc=0.9569 f1=0.9654 auc=0.9989

AdaBoost
 WITH BHO    | acc=0.9670 f1=0.9737 auc=0.9986
 WITHOUT BHO | acc=0.9670 f1=0.9737 auc=0.9989

GradBoost
 WITH BHO    | acc=0.9619 f1=0.9697 auc=0.999
 WITHOUT BHO | acc=0.9645 f1=0.9717 auc=0.9996

XGBoost
 WITH BHO    | acc=0.9619 f1=0.9696 auc=0.9995
 WITHOUT BHO | acc=0.9645 f1=0.9717 auc=0.9998
